source: https://www.bea.gov/data/gdp/gross-domestic-product

Creating Vintage Time Series of BEA estimations and revisions of US GDP.

In [ ]:
import numpy as np
import pandas as pd
import re
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
import matplotlib.dates as mdates
from IPython.display import HTML
import sys 
import ipywidgets as widgets
from IPython.display import display
import shutil
shutil.which("ffmpeg")

In [ ]:
# Publication Date and readme info
# --- Extract text info (first 5 rows from Vintage History) ---
text_info_df = pd.read_excel(
    'gdp-gdi-vintage-history.xlsx', 
    sheet_name='Vintage History',
    header=None,
    nrows=4)

# Get first 5 rows and convert everything to string
text_info_df = text_info_df.astype(str)
text_info_df = text_info_df.fillna("")

# Flatten into a list of strings (optional)
text_info_strings = text_info_df.values.flatten().tolist()

# Remove 'nan' strings if needed
text_info_strings = [s for s in text_info_strings if s.lower() != 'nan']

for item in text_info_strings:
    print(item)

In [ ]:
# --- Extract read_me sheet (Sheet 2) ---
readme = pd.read_excel( 'gdp-gdi-vintage-history.xlsx', 
    sheet_name='ReadMe', 
    header=None)

# Convert all to strings
readme_strings = readme.astype(str)

# Flatten 
readme_list = readme_strings.values.flatten().tolist()

# Remove empty/nan
readme_list = [s for s in readme_list if s.lower() != 'nan']

for item in readme_list:
    print(item)


In [ ]:
# Download current version of the excel file from
# https://www.bea.gov/sites/default/files/2026-02/gdp-gdi-vintage-history.xlsx
# read excel file from sheet 'Vintage History' into data frame 'raw'
raw  = pd.read_excel('gdp-gdi-vintage-history.xlsx', sheet_name='Vintage History',
    skiprows=5,
    header=None)

Table needs to be cleaned:
- each quarter with its' estimations and revisions is listed in an individual table
    - Identify boundries of the tables using column (0)
- Create meaningfull headers
- Clean numerical data
- Clean datetime data
- Create Valid From and Valid to columns

In [ ]:
# For each Quarter there is an individual table
# Fill nan in first column with new column to create quarter date
# New column name
raw['Quarter'] = raw[0]
# Down fill 'Quarter'
raw['Quarter'] = raw['Quarter'].ffill()

In [ ]:
# Transfer to Dataframe 
df = raw[[ 'Quarter', 1, 2, 3, 4, 5, 6]].copy()

In [ ]:
# Assign column names
df.columns = [
    "Quarter",
    "Type",
    "GDP",
    "GDI",
    "GDP_%_Change",
    "GDI_%_Change",
    "Release_Date"
]

In [ ]:
# Split Release_Date into Release_Date and Note 
df['Release_Date'] = df['Release_Date'].astype(str)
df['Note'] = df['Release_Date'].str[12:].str.strip().replace("",pd.NA)
df['Release_Date'] = df['Release_Date'].str[:12]

In [ ]:
# Change Release_Date into Date time
df['Release_Date'] = pd.to_datetime(
    df['Release_Date'],
    format="%b %d, %Y",
    errors='coerce'
)

In [ ]:
# Drop NaT Release_Date. These Rows are not needed
df = df.dropna(subset=['Release_Date'])

In [ ]:
# Change Data types of 'GDP','GDI','GDP_%_Change','GDI_%_Change'
cols = ['GDP','GDI','GDP_%_Change','GDI_%_Change']
df[cols] = (
    df[cols]
    .replace(".....", pd.NA) # handle missing marker
    .replace(",", "", regex=True)     # remove thousands separators
    .apply(pd.to_numeric, errors="coerce")
)

In [ ]:
# Create Quarter_Date column from Quarter.  
df['Quarter_Date'] = pd.PeriodIndex(df['Quarter'], freq='Q').to_timestamp(how='end').normalize()

In [ ]:
# Reset Index
df = df.reset_index(drop=True)

In [ ]:
# Basic statistics
df.describe()

In [ ]:
# Latest Release_Date for each Quarter
df['Last_Release_Date'] = (
    df.groupby('Quarter')['Release_Date'].transform('max')
)

In [ ]:
# Create Valid_From and Valid_To
# First check if df is sorted via 'Quarter_Date', 'Release_Date'
# If is_sorted is True then Valid_From and Valid_To can be created
is_sorted = (
    df.groupby('Quarter_Date')['Release_Date']
    .apply(lambda s: s.is_monotonic_decreasing)
    .all()
)
is_sorted	   

In [ ]:
# Extra Check for possible quaters that are not sorted if is_sorted is False 
bad_quarters = (
    df.groupby("Quarter")["Release_Date"]
      .apply(lambda s: not s.is_monotonic_decreasing)
)
for item in bad_quarters:
    print(item)

In [ ]:
# Create Valid_From and Valid_To
df['Valid_From'] = df['Release_Date']
df['Valid_To'] = (df.groupby('Quarter')['Release_Date'].shift(+1))

In [ ]:
# Verify Validity Intervals Per Quarter
test = (
    df.sort_values(['Quarter_Date', 'Release_Date'])
      .groupby('Quarter')
      .apply(lambda x: (x['Valid_From'].shift(-1) < x['Valid_To']).any())
)
print(test[test])

In [ ]:
df[['Quarter_Date', 'GDP']].loc[df['Release_Date'] == '2023-09-28']

In [ ]:
df.dtypes

In [ ]:
df.head(10)

In [ ]:
# Which quarters have been changed during the last release date?
# previous estimate within each quarter
df["GDP_%_Change_Previous"] = df.groupby("Quarter")["GDP_%_Change"].shift(-1)
df["GDP_Previous"] = df.groupby("Quarter")["GDP"].shift(-1)
df["GDI_Previous"] = df.groupby("Quarter")["GDI"].shift(-1)
df["GDI_%_Change_Previous"] = df.groupby("Quarter")["GDI_%_Change"].shift(-1)

In [ ]:
# Changes in the last release date
df[
    ['Release_Date','Quarter','Type','Note', 'GDP','GDP_%_Change','GDP_Previous','GDP_%_Change_Previous', 'GDI','GDI_%_Change','GDI_Previous','GDI_%_Change_Previous']
    ].loc[
        df['Release_Date'] == df['Release_Date'].max()
        ]

In [ ]:
#df[
#    ['Quarter','Type','Note', 'GDP','GDP_%_Change','GDP_Previous','GDP_%_Change_Previous', 'GDI','GDI_%_Change','GDI_Previous','GDI_%_Change_Previous']
#    ].loc[
#        df['Release_Date'] == '2023-09-28'
#        ].head(10)

In [ ]:
# Which quarter has had the most revisions?
quarters_counts = df['Quarter'].value_counts()
quarters_counts[quarters_counts == quarters_counts.max()]

In [ ]:
quarters_counts.head(5)

In [ ]:
# At which release date were the most quarters revised?
releasedates_counts = df['Release_Date'].value_counts()
releasedates_counts[releasedates_counts == releasedates_counts.max()]

In [ ]:
releasedates_counts.head(5)

In [ ]:
# Aggregated Results of Quarter estimations/calculations
df_stat = df.groupby('Quarter')[cols].agg(['min','max','mean','std'])

df_stat.head(10)

In [ ]:
# Calculate GDP % Change Switch
df_stat['GDP_%_Change_Switch']  = (df_stat[('GDP_%_Change', 'min')] * df_stat[('GDP_%_Change', 'max')]) < 0

In [ ]:
# List those quarters were an estimation of economic contraction has changed to growth or vice versa
df_stat[df_stat['GDP_%_Change_Switch'] == True]

In [ ]:
# Calculate diff of max and min gdp change
df_stat['Abs_Change_GDP'] = abs(df_stat[('GDP_%_Change', 'max')] - df_stat[('GDP_%_Change', 'min')])


In [ ]:
# Transform to present the biggest changes
df_out = df_stat.reset_index()
df_out = df_out[[('Quarter',''),('GDP_%_Change', 'max'),('GDP_%_Change', 'min'),('Abs_Change_GDP','')]]
df_out.columns = [
    'Quarter',
    'GDP_Max',
    'GDP_Min',
    'Abs_Change_GDP'
] 
df_out = df_out.sort_values('Abs_Change_GDP', ascending=False)

In [ ]:
# What is the largest revisions?
df_out[df_out['Abs_Change_GDP'] == df_out['Abs_Change_GDP'].max()]

In [ ]:
# What are the largest revisions?
df_out.head(10)

In [ ]:
# Build time series using validity intervals
# Use row where as_of >= Valid_From and (as_of <= Valid_To OR Valid_To is NaT)
def build_vintage_timeseries(df):
    value_cols = [
        'GDP',
        'GDI',
        'GDP_%_Change',
        'GDI_%_Change'
    ]

    # List of Unique Release_Date
    release_dates = (
        df['Release_Date']
        .dropna()
        .sort_values()
        .unique()
    )

    results ={}

    # Filling Results
    for as_of in release_dates:
        snapsh = df[
            (df['Valid_From'] <= as_of) &
            (
                (df['Valid_To'] > as_of) |
                df['Valid_To'].isna()
                  )
        ]

        snapsh = (
            snapsh[
                ['Quarter', 'Quarter_Date'] + value_cols
            ]
            .sort_values('Quarter_Date')
            .reset_index(drop=True)
        )

        results[pd.Timestamp(as_of)] = snapsh

    return results

In [ ]:
# The time series are transfered as data frames into a dictionary
vint = build_vintage_timeseries(df)
list(vint.keys())[:10]

In [ ]:
# Alternatively transfer of the time series into single data frame 
panel = (
    pd.concat(vint, names=["As_Of_Release_Date"])
      .reset_index(level=0)
)

In [ ]:
# unique dates
unique_dates = sorted(df['Release_Date'].unique())

In [ ]:
# Function to recieve most recent date
def get_most_recent_date(input_date, date_list):
    input_date = pd.to_datetime(input_date)
    valid_dates = [d for d in date_list if d <= input_date]
    
    if not valid_dates:
        return None  # or raise ValueError("No valid date found")
    
    return max(valid_dates)

In [ ]:
pd.Timestamp(get_most_recent_date('2006-12-31', unique_dates))

In [ ]:
df_temp = pd.DataFrame()
as_of = get_most_recent_date('2034-12-31', unique_dates)
# df_temp = vint[as_of]
df_temp = vint.get(pd.Timestamp(as_of))
as_of

In [ ]:
# plot Gdp and GDP_%_Change and specific timespot
plt.figure()
ax1 = plt.gca()
# First graph: line chart for GDP
ax1.plot(df_temp["Quarter_Date"], df_temp["GDP"], color='blue')
ax1.set_xlabel("Quarter")
ax1.set_ylabel("GDP", color='blue')
ax1.tick_params(axis='y', labelcolor='blue')

# Second graph: bar chart for GDP % Change
ax2 = ax1.twinx()
ax2.bar(df_temp["Quarter_Date"], df_temp["GDP_%_Change"], color='red', alpha=0.5, width=20)
ax2.set_ylabel("GDP_%_Change", color='red')
ax2.tick_params(axis='y', labelcolor='red')

plt.title(f'GDP (bil dol) and GDP % Change as of {as_of.date()}')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# Plot ALL vintages on one chart (GDP only)
plt.figure()
for rd,  group in panel.groupby('As_Of_Release_Date'):
    plt.plot(group['Quarter_Date'], group['GDP'])
plt.xlabel("Quarter")
plt.ylabel("GDP")
plt.title(f'ALL vintages on one chart (GDP only)')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# Plot one Quarter GDP across vintages (revision path)
quarter = '2002Q4'
revision_df = panel[panel['Quarter'] == quarter].sort_values('As_Of_Release_Date')
plt.figure()
plt.plot(revision_df["As_Of_Release_Date"],
         revision_df["GDP"])
plt.xlabel("Release Date")
plt.ylabel("GDP")
plt.title(f'Revision path of GDP {quarter}')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# Plot one Quarter GDP Growth Change across vintages (revision path)
quarter = '2008Q1'
revision_df = panel[panel['Quarter'] == quarter].sort_values('As_Of_Release_Date')
plt.figure()
plt.plot(revision_df["As_Of_Release_Date"],
         revision_df["GDP_%_Change"])
plt.xlabel("Release Date")
plt.ylabel("GDP")
plt.title(f'Revision path of GDP Growth {quarter}')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# Animate GDP Vintage Evolution (matplotlib)

# Sort release dates
release_dates = sorted(vint.keys())
fig, ax = plt.subplots(figsize=(10, 6))
line, = ax.plot([], [], lw=2)

ax.set_xlabel("Quarter")
ax.set_ylabel("GDP")
ax.set_title("GDP Vintage Evolution")

def init():
    ax.set_xlim(
        min(df['Quarter_Date']),
        max(df['Quarter_Date'])
    )
    ax.set_ylim(
        df['GDP'].min(),
        df['GDP'].max()
    )
    return line,

def update(frame):
    as_of = release_dates[frame]
    snap = vint[as_of]
    
    line.set_data(
        snap['Quarter_Date'],
        snap['GDP']
    )
   
    ax.set_title(f"GDP Vintage as of {as_of.date()}")
    return line,

ani = FuncAnimation(
    fig,
    update,
    frames=len(release_dates),
    init_func=init,
    blit=True,
    interval=300
)
# Save as GIF
ani.save("gdp_animation.gif", writer="pillow", fps=3)
HTML(ani.to_jshtml())
# plt.show()


In [ ]:
# Animate GDP growth Vintage Evolution (matplotlib)

# Sort release dates
release_dates = sorted(vint.keys())
fig, ax = plt.subplots(figsize=(10, 6))
line, = ax.plot([], [], lw=2)

ax.set_xlabel("Quarter")
ax.set_ylabel("GDP Growth")
ax.set_title("GDP Growth Vintage Evolution")

def init_growth():
    ax.set_xlim(
        min(df['Quarter_Date']),
        max(df['Quarter_Date'])
    )
    ax.set_ylim(
        df['GDP_%_Change'].min(),
        df['GDP_%_Change'].max()
    )
    return line,

def update_growth(frame):
    as_of = release_dates[frame]
    snap = vint[as_of]
    
    line.set_data(
        snap['Quarter_Date'],
        snap['GDP_%_Change']
    )
   
    ax.set_title(f"GDP growth Vintage as of {as_of.date()}")
    return line,

ani = FuncAnimation(
    fig,
    update_growth,
    frames=len(release_dates),
    init_func=init_growth,
    blit=True,
    interval=300
)
# Save as GIF
ani.save("gdp_growth_animation.gif", writer="pillow", fps=3)
HTML(ani.to_jshtml())

In [ ]:
# Animate GDP growth Vintage Evolution (BAR CHART)

release_dates_bar = sorted(vint.keys())

fig, ax = plt.subplots(figsize=(12, 6))

ax.set_xlabel("Quarter")
ax.set_ylabel("GDP Growth")


def get_bar_widths(dates):
    """
    Each bar spans from previous quarter to current quarter.
    Last bar gets normal quarter width.
    """
    d = pd.to_datetime(dates).sort_values().reset_index(drop=True)

    widths = []

    for i in range(len(d)):
        if i == 0:
            if len(d) > 1:
                w = (d.iloc[1] - d.iloc[0]).days
            else:
                w = 90
        else:
            w = (d.iloc[i] - d.iloc[i - 1]).days

        widths.append(w)

    # ensure final bar visible
    if len(d) > 1:
        widths[-1] = widths[-2]

    return np.array(widths)


def update_growth(frame):
    ax.clear()

    as_of = release_dates_bar[frame]
    snap = vint[as_of].copy().sort_values("Quarter_Date")

    dates = pd.to_datetime(snap["Quarter_Date"])
    vals = snap["GDP_%_Change"].values
    widths = get_bar_widths(dates)

    # Start bars at left edge = previous quarter boundary
    left_edges = dates - pd.to_timedelta(widths, unit="D")

    ax.bar(
        left_edges,
        vals,
        width=widths,
        align="edge",
        alpha=0.75
    )

    # Extend xlim so final bar fully visible
    xmax = dates.iloc[-1] + pd.Timedelta(days=widths[-1])

    ax.set_xlim(dates.iloc[0] - pd.Timedelta(days=widths[0]), xmax)

    # Dynamic y limits
    ymin = vals.min()
    ymax = vals.max()
    pad = (ymax - ymin) * 0.1 if ymax != ymin else 1

    ax.set_ylim(ymin - pad, ymax + pad)

    ax.set_xlabel("Quarter")
    ax.set_ylabel("GDP Growth")
    ax.set_title(f"GDP Growth Vintage as of {as_of.date()}")

    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
    plt.setp(ax.get_xticklabels(), rotation=45, ha="right")

    return ax.patches


ani = FuncAnimation(
    fig,
    update_growth,
    frames=len(release_dates),
    interval=300,
    blit=False
)

ani.save("gdp_growth_bar_animation.gif", writer="pillow", fps=3)

HTML(ani.to_jshtml())

In [ ]:
# Prepare release dates (ensure consistent type)
release_dates = sorted(pd.to_datetime(list(vint.keys())))

In [ ]:
# Date picker for UI
date_picker = widgets.DatePicker(
    description='Pick a date:',
)

In [ ]:
# Create plot with UI
def plot_for_selected_date(picked_date):
    if picked_date is None:
        return
    
    # Use your function here
    closest_date = get_most_recent_date(picked_date, release_dates)
    
    if closest_date is None:
        print("No available data before this date.")
        return
    
    snap = vint[closest_date]
    
    plt.figure(figsize=(10, 6))
    plt.plot(snap['Quarter_Date'], snap['GDP_%_Change'], color="blue")
    plt.title(f"GDP as of {closest_date.date()} (selected {picked_date})")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()



In [ ]:
widgets.interactive(plot_for_selected_date, picked_date=date_picker)